# 01 — Numerical Stability & Perplexity**Time: ~30 minutes. Scope: one concept, done properly.**This notebook works through a classic failure: a sequence-scoring function that breaks on long inputs.You will:1. Reproduce the bug yourself (a scoring function that returns `inf` on long sequences)2. Diagnose it — floating-point underflow3. Fix it with log-space arithmetic, which solves the crash *and* the speed problem4. See why perplexity matters to you specifically as a red teamer**Why this one first:** perplexity is not an ML-researcher curiosity. It is thecore signal behind AI-generated-text detection, membership inference attacks, andtraining-data extraction. You have read papers that use it. Now you will own it.

---## How to use a Jupyter notebook (60-second version)A notebook is a list of **cells**. Two kinds matter:- **Markdown cells** — text, like this one- **Code cells** — Python that actually runs**Run a cell:** click it, press `Shift + Enter`. It executes and moves to the next cell.**The one thing that trips everyone up:** the notebook holds **state**. Variablesdefined in one cell stay alive for every cell after it. So order matters — butonly the order you *ran* things, not the order they appear on screen. If somethingbehaves strangely, go to `Kernel → Restart & Run All` and start clean.**Why use a notebook instead of a `.py` script:** you can run one piece, look at theresult, change it, and run it again without re-running everything. It is a labbench, not a program. That makes it the right tool for *understanding* code andfor analysis — and the wrong tool for shipping software.**Run the cells in order, top to bottom.**

In [ ]:
import numpy as np
import warnings

print("numpy", np.__version__)

---## Part 1 — Reproduce the bugBelow is a stand-in language model. Real models return the probability of eachtoken given everything before it. Ours just makes up plausible numbers, which isall we need — the bug has nothing to do with the model.Note the probabilities are small. That is realistic: a model choosing from a50,000-token vocabulary assigns most tokens a low probability.

In [ ]:
class MyLanguageModel:
    """Stand-in for a real language model.

    get_probabilities(seq) returns p_i = P(token_i | all tokens before it).
    Real values are small — most tokens are individually unlikely.
    """
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def get_probabilities(self, seq):
        return list(self.rng.uniform(0.001, 0.3, len(seq)))


model = MyLanguageModel(seed=1)
sample = model.get_probabilities(["the", "cat", "sat"])
print(sample)

### The original function, exactly as it appearedRead the docstring. It computes a **measure of surprise**: multiply every tokenprobability together to get the probability of the whole sequence, take thegeometric mean, invert it. Big number = the model finds this text weird.This is **perplexity**, though the code never says so.

In [ ]:
def sequence_score_naive(model, seq):
    """Naive version. Contains the bug."""
    # p_i = P(t_i | t_0 ... t_i-1)
    p = model.get_probabilities(seq)

    # probability of the entire sequence
    prob = 1.0
    for i in range(len(p)):
        prob *= p[i]

    # geometric mean
    mean_prob = prob ** (1 / len(p))

    # inverse
    return 1 / mean_prob

### Short sequence — works fine

In [ ]:
m = MyLanguageModel(seed=1)
print("100 tokens:", sequence_score_naive(m, ["tok"] * 100))

### Long sequence — the reported bugThe ticket said: *anything over 500 or so tokens returns infinity, and logs`RuntimeWarning: divide by zero encountered in scalar divide`.*

In [ ]:
m = MyLanguageModel(seed=1)
print("500 tokens:", sequence_score_naive(m, ["tok"] * 500))

---## Part 2 — Diagnose itThe warning says *divide by zero*. The only division is `1 / mean_prob`.So `mean_prob` must be exactly `0.0`. Work backwards: `mean_prob` is`prob ** (1/n)`, and `0.0 ** anything` is `0.0`. So **`prob` itself is zero.**Why? Every probability is below 1. Multiply 500 of them together and you getsomething like 10⁻⁵⁰⁰.Now look at what a 64-bit float can actually hold.

In [ ]:
p = np.random.default_rng(3).uniform(0.001, 0.3, 500)

print("product of 500 probabilities:", np.prod(p))
print()
print("smallest normal float64:   ", np.finfo(np.float64).tiny)
print("smallest subnormal float64:", np.nextafter(0, 1))

**There it is.** A float64 bottoms out around `1e-308`. Anything smallerrounds to exactly zero. This is called **underflow**.Watch it happen progressively — the product collapses to zero somewherebetween 300 and 500 tokens depending on the values.

In [ ]:
for n in [50, 100, 200, 300, 400, 500, 600]:
    p = np.random.default_rng(3).uniform(0.001, 0.3, n)
    print(f"n={n:4d}  product = {np.prod(p):.3e}")

---## Part 3 — The trap in the hintThe obvious first suggestion is to swap the loop for `np.product()`.That is a real optimization — replacing the Python `for` loop with a vectorizedNumPy call is much faster. **But it does not fix the infinity.** NumPy underflowsto zero exactly like the loop does.Catching that distinction — *this hint solves half the problem* — is worth morethan producing the fix. Two bugs were reported. One fix addresses one of them.

In [ ]:
p = np.random.default_rng(3).uniform(0.001, 0.3, 500)

loop_result = 1.0
for x in p:
    loop_result *= x

print("python loop:", loop_result)
print("np.prod:   ", np.prod(p))
print("same answer, same underflow — only the speed changed")

---## Part 4 — The fix: log spaceThe move that solves **both** problems at once.The identity that makes it work:$$\log(a \times b) = \log(a) + \log(b)$$So instead of multiplying 500 tiny numbers, **add their logarithms**. The log ofa tiny number is just a moderately negative number — `log(1e-300)` is about`-690`, which a float64 handles without blinking.Rewriting the whole formula in log space:$$\text{score} = \frac{1}{\left(\prod_i p_i\right)^{1/n}} = \exp\!\left(-\frac{1}{n}\sum_i \log p_i\right)$$The right-hand side never computes a tiny number at any point.

In [ ]:
def sequence_score_log(model, seq):
    """Log-space version. Vectorized (fast) and underflow-proof (correct)."""
    p = np.asarray(model.get_probabilities(seq), dtype=np.float64)
    return float(np.exp(-np.log(p).mean()))

### Check 1 — does it agree with the original where the original works?Always verify a rewrite against the version you trust, on inputs where the oldone is still valid. This is the habit that separates a fix from a guess.

In [ ]:
a = sequence_score_naive(MyLanguageModel(seed=1), ["t"] * 50)
b = sequence_score_log(MyLanguageModel(seed=1), ["t"] * 50)

print("naive:", a)
print("log:  ", b)
print("match:", np.isclose(a, b))

### Check 2 — does it survive the case that broke the original?

In [ ]:
for n in [500, 5000, 50000]:
    score = sequence_score_log(MyLanguageModel(seed=7), ["t"] * n)
    print(f"n={n:6d}  score = {score:.4f}")

### Check 3 — the speed problemThe second complaint was that millions of sequences took hours.

In [ ]:
P = np.random.default_rng(9).uniform(0.001, 0.3, 1000)

def loop_version(p):
    prob = 1.0
    for i in range(len(p)):
        prob *= p[i]
    return prob

def log_version(p):
    return np.exp(-np.log(p).mean())

In [ ]:
# %%timeit is a Jupyter "magic" command — it runs a line many times and reports the average.
%timeit loop_version(P)

In [ ]:
%timeit log_version(P)

Roughly an order of magnitude, and the gap widens with sequence length. ThePython loop interprets one multiplication at a time; NumPy pushes the wholearray through compiled code.**One fix, both bugs.**

---## Part 5 — The edge case that bites in production`log(0)` is negative infinity. If a model ever returns a probability of exactlyzero — and quantized or heavily-filtered models do — the fix fails in a new way.Two defenses, in order of preference:1. **Ask the model for log-probabilities directly.** Most APIs expose them   (`logprobs`). Taking the log of an already-rounded float throws away precision   that was never recoverable. This is the strongest answer, and raising it in a   code review signals you have thought past the immediate code.2. **Clamp**, if you only have probabilities.

In [ ]:
pz = np.array([0.5, 0.0, 0.2])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print("raw log:    ", np.log(pz))

print("clamped log:", np.log(np.clip(pz, 1e-12, 1.0)))

In [ ]:
def sequence_score_final(model, seq, floor=1e-12):
    """Production-shaped version."""
    p = np.asarray(model.get_probabilities(seq), dtype=np.float64)
    if p.size == 0:
        raise ValueError("empty sequence")
    logp = np.log(np.clip(p, floor, 1.0))
    return float(np.exp(-logp.mean()))


print(sequence_score_final(MyLanguageModel(seed=2), ["t"] * 2000))

---## Part 6 — Why you care: perplexity is a security signalThis is the part that makes the exercise yours rather than an ML homework problem.**Connection 1 — perplexity is the loss curve.**The training loss for a language model is **cross-entropy**, which is`-mean(log p)` — the exact quantity in the middle of your fix. Perplexity is just`exp(cross_entropy)`. So the squiggly loss curve you were shown and the functionyou just debugged are the same number, one exponentiated. Learn one, you have both.**Connection 2 — membership inference.**Models are **more confident on data they were trained on**. Lower loss, lowerperplexity. That gap is the entire basis of membership inference attacks: score acandidate record, and if perplexity is suspiciously low, it was probably in thetraining set. Extend it and you get training-data extraction.**Connection 3 — AI-generated-text detection.**Text sampled from a model is, by construction, unsurprising *to that model*.Low perplexity. Most detectors are a threshold on this signal — which is also whythey are evadable, and why they misfire on non-native English writers whosephrasing is more predictable.Let's make Connection 2 visible.

In [ ]:
rng = np.random.default_rng(11)

# Simulated perplexity for records the model trained on vs. records it never saw.
member_ppl    = np.exp(rng.normal(1.6, 0.35, 500))   # seen in training
nonmember_ppl = np.exp(rng.normal(2.4, 0.45, 500))   # never seen

print("median perplexity, members:    ", round(np.median(member_ppl), 2))
print("median perplexity, non-members:", round(np.median(nonmember_ppl), 2))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 40, 60)
ax.hist(member_ppl, bins=bins, alpha=0.6, label="in training set")
ax.hist(nonmember_ppl, bins=bins, alpha=0.6, label="not in training set")
ax.axvline(8.0, color="k", linestyle="--", linewidth=1, label="attacker threshold")
ax.set_xlabel("perplexity")
ax.set_ylabel("count")
ax.set_title("Membership inference: the whole attack in one picture")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
threshold = 8.0

tpr = (member_ppl < threshold).mean()
fpr = (nonmember_ppl < threshold).mean()

print(f"Guess 'was in training set' when perplexity < {threshold}")
print(f"  true positive rate:  {tpr:.0%}")
print(f"  false positive rate: {fpr:.0%}")
print()
print("The distributions overlap — which is why real membership inference is")
print("noisy per-record, and why it is reported as an AUC over many records")
print("rather than a verdict on any single one.")

---## What you can now say out loudNot memorized lines — things that are now true:- *"That function is perplexity. The bug is float underflow: multiply 500  probabilities and you fall below 1e-308, which rounds to zero, then you divide  by it. Vectorizing fixes the speed but not the crash — the fix for both is log  space, summing log-probabilities instead of multiplying."*- *"Cross-entropy loss and perplexity are the same quantity, one exponentiated."*- *"Membership inference is a threshold on that signal — models are more  confident on their training data."*- *"If the API exposes logprobs, take them directly rather than logging a  rounded probability."*---## Extensions — only if you want them- Swap the uniform stand-in for a real model (`transformers`, small GPT-2) and  compute perplexity on text you wrote vs. text a model generated.- Implement `logsumexp` by hand and work out why the max-subtraction trick  prevents overflow. Same family of problem, other direction.- Score a handful of well-known public strings and see which are lowest-perplexity.**Stop here if the 30 minutes are up.** The core concept is already yours.